In [ ]:
%run "00_globals_and_db.ipynb"


In [ ]:
# ---------------------------------------------------------
# SETUP AND CONFIGURATION
# ---------------------------------------------------------

# Import a custom web scraping session manager. This helps download web pages.
from scrapling.fetchers import FetcherSession

# Load a JSON file containing a list of years and their corresponding URLs.
# json.loads converts the text from the file into a Python list/dictionary.
years = json.loads((DIR_RAW / "years.json").read_text(encoding="utf-8"))

VVTAT_BASE = "https://vvtat.lrv.lt"

# These are keywords we are looking for in links to find specific categories of cases.
GOODS_HINTS = ["gincai-del-prekiu", "ginčai dėl prekių", "gincai del prekiu"]
SERV_HINTS  = ["gincai-del-paslaugu", "ginčai dėl paslaugų", "gincai del paslaugu"]

# MONTH_RE looks for a year and month in a URL (e.g., /2026-01/).
MONTH_RE = re.compile(r"/(20\d{2})-(\d{2})/")    # e.g. /2026-01/

# PDF_RE looks for ".pdf" at the end of a link.
PDF_RE = re.compile(r"\.pdf(\?|$)", re.IGNORECASE)

# CASE_NO_RE looks for the letters "Nr." followed by a mix of numbers and letters (the case number).
CASE_NO_RE = re.compile(r"Nr\.\s*([0-9A-Za-zĄČĘĖĮŠŲŪŽąčęėįšųūž\-\/]+)", re.IGNORECASE)

In [ ]:
# ---------------------------------------------------------
# HELPER FUNCTIONS
# ---------------------------------------------------------
def find_category_links(page):
    #The -> list[dict] part in the original code is called a Type Hint.
    # This function scans a web page for links related to "goods" or "services".
    
    goods = None
    serv = None

    # page.css("a[href]") finds all clickable links (<a> tags) on the page.
    for a in page.css("a[href]"):
        href = a.attrib.get("href", "") # Gets the actual URL
        txt = norm_text(a.text).lower() # Gets the visible text of the link
        href_l = href.lower()

        # If the URL or the text contains our keywords, we save the full URL.
        # urljoin combines a base web address with a relative link to make a complete URL.
        # VVTAT_BASE is defined in the setup cell above so this notebook also works on its own.
        if any(h in href_l for h in GOODS_HINTS) or any(h in txt for h in GOODS_HINTS):
            goods = urljoin(VVTAT_BASE, href)
        if any(h in href_l for h in SERV_HINTS) or any(h in txt for h in SERV_HINTS):
            serv = urljoin(VVTAT_BASE, href)

    # Return the found links as a list of dictionaries.
    out = []
    if goods:
        out.append({"case_type": "goods", "url": goods})
    if serv:
        out.append({"case_type": "services", "url": serv})
    return out

In [ ]:
#safety function, if vvtat page starts to pu the data into monthly category pages
def discover_month_pages(page, year):
    # This function finds links to sub-pages organized by month.
    urls = set() # A 'set' automatically prevents duplicate URLs.

    # Always include the page we are currently looking at.
    if hasattr(page, "url") and page.url:
        urls.add(page.url)

    # Look at all links on the page.
    for a in page.css("a[href]"):
        #'a' represents one specific link at a time. 
        # .attrib.get("href", "") pulls the destination URL out of that link.
        # If for some reason the link is empty, it gives us an empty string ("").
        
        href = a.attrib.get("href", "")
        if not href:
            continue
        
        # urljoin() attaches that to the main website address (like "https://vvtat.lt")
        # to create a "full" working link that you could actually paste into a browser.
        full = urljoin(VVTAT_BASE, href)
        
        # Check if the link matches month pattern (e.g., /2026-05/) and matches the current year.
        m = MONTH_RE.search(full)
        if m and int(m.group(1)) == year:
            urls.add(full)

    # Return the unique URLs sorted in alphabetical order.
    return sorted(urls)

In [ ]:
def find_updated_at_anywhere(page):
    # This function searches the entire raw HTML text for the phrase "Atnaujinimo data:" 
    # (Update date) followed by a date in YYYY-MM-DD format.
    html = page.text if hasattr(page, "text") else ""
    m = re.search(r"Atnaujinimo data:\s*(\d{4}-\d{2}-\d{2})", html)
    # If found, it parses the date; otherwise, it returns None.
    return parse_iso_date(m.group(1)) if m else None

In [ ]:
def extract_case_rows_structure(page, source_list_url, year, case_type):
    # This is the heavy lifter. It extracts the actual data from a table on the web page.
    updated_at = find_updated_at_anywhere(page)
    rows = []

    # Iterate through every table row ("tr") on the page.
    for tr in page.css("tr"):
        tds = tr.css("td") # Find all table data cells ("td") in this row.
        
        # If the row has less than 2 columns, it's not a valid data row, so skip it.
        if len(tds) < 2:
            continue

        # Look for a PDF link inside this row.
        pdf_a = None
        for a in tr.css("a[href]"):
            href = a.attrib.get("href", "")
            if href and PDF_RE.search(href): # If the link ends in .pdf
                pdf_a = a
                break
        
        # If there's no PDF link, skip this row.
        if not pdf_a:
            continue

        # Extract the PDF URL and the text of the link.
        pdf_url = urljoin(VVTAT_BASE, pdf_a.attrib.get("href", ""))
        link_text = norm_text(pdf_a.text)

        # Extract text from the first column (Company) and second column (Subject).
        company = norm_text(tds[0].get_all_text() if hasattr(tds[0], "get_all_text") else tds[0].text)
        subject = norm_text(tds[1].get_all_text() if hasattr(tds[1], "get_all_text") else tds[1].text)

        # Try to guess the case number from the link text using our regex.
        case_no_guess = None
        m_case = CASE_NO_RE.search(link_text)
        if m_case:
            case_no_guess = m_case.group(1)

        # Try to guess the decision date from the link text (looking for YYYY-MM-DD).
        decision_date_guess = None
        m_iso = re.match(r"^(20\d{2}-\d{2}-\d{2})\b", link_text)
        if m_iso:
            decision_date_guess = parse_iso_date(m_iso.group(1))

        # Bundle all this extracted data into a dictionary (like a single record/row).
        rows.append({
            "year": year,
            "case_type": case_type,
            "source_list_url": source_list_url,
            "list_page_updated_at": updated_at,
            "company_name_raw": company,
            "subject_raw": subject,
            "Ginčo dalykas": subject,
            "pdf_url": pdf_url,
            "pdf_url_hash": sha256_text(pdf_url), # Creates a unique fingerprint for this PDF
            "link_text": link_text,
            "case_no_guess": case_no_guess,
            "decision_date_guess": decision_date_guess,
        })

    # Remove duplicates based on the PDF's unique fingerprint.
    uniq = {}
    for r in rows:
        uniq[r["pdf_url_hash"]] = r
    
    # Return the deduplicated list of rows.
    return list(uniq.values())

In [ ]:
# ---------------------------------------------------------
# THE MAIN SCRAPING LOOP
# ---------------------------------------------------------

# Define where to save the final JSON Lines (.jsonl) file.
out_jsonl = DIR_RAW / "case_rows.jsonl"

# Connect to the database and create a cursor to execute SQL commands.
cnx = db_connect()
cur = cnx.cursor()

count = 0

# Open a web scraping session and open the output file for writing ("w").
# The 'with' statement ensures these close properly when the code finishes.
with FetcherSession(timeout=30, retries=3) as session, out_jsonl.open("w", encoding="utf-8") as f:
    
    # Loop through every year defined in the years.json file.
    for y in years:
        year = int(y["year"])
        year_url = y["url"]

        # Download the main page for that year.
        year_page = session.get(year_url)
        if year_page.status != 200: # 200 means "OK/Success" in HTTP status codes.
            print("WARN year page failed:", year, year_page.status)
            continue

        # Find the "goods" and "services" links on that year's page.
        cats = find_category_links(year_page)
        if not cats:
            print("WARN no categories found for year:", year, year_url)
            continue

        # Loop through the found categories (goods and services).
        for cat in cats:
            case_type = cat["case_type"]
            cat_url = cat["url"]

            # Download the category page.
            cat_page = session.get(cat_url)
            if cat_page.status != 200:
                print("WARN category page failed:", cat_url, cat_page.status)
                continue

            # Save the raw, unedited HTML of the page to the database.
            upsert_raw_page(cur, cat_url, cat_page.status, cat_page.body, extracted={"year": year, "case_type": case_type})
            cnx.commit() # Save changes to the database.

            # Find if there are specific pages for different months.
            month_urls = discover_month_pages(cat_page, year)

            # Loop through every month page found.
            for list_url in month_urls:
                lp = session.get(list_url) # Download the month page.
                if lp.status != 200:
                    continue

                # Save the raw HTML of the month page to the database.
                upsert_raw_page(cur, list_url, lp.status, lp.body, extracted={"year": year, "case_type": case_type})
                cnx.commit()

                # Extract the actual data rows from the table on this page.
                rows = extract_case_rows_structure(lp, list_url, year, case_type)

                # Save each extracted row.
                for r in rows:
                    # Save the row to the database.
                    row_id = upsert_raw_case_row(cur, r)
                    cnx.commit()

                    # Add the new database ID to our record.
                    r2 = dict(r)
                    r2["row_id"] = row_id
                    
                    # Write the record to our JSON Lines file.
                    # json.dumps turns the Python dictionary back into text.
                    f.write(json.dumps(r2, ensure_ascii=False, default=str) + "\n")
                    count += 1

                # Pause randomly between 2 and 5 seconds. This mimics human behavior 
                # and prevents the website from blocking you for scraping too fast.
                sleep_human(2, 5)
            
            # Pause between categories.
            sleep_human(2, 5)
        
        # Pause between years.
        sleep_human(2, 5)


# ---------------------------------------------------------
# CLEANUP
# ---------------------------------------------------------

# Always close database cursors and connections when finished to free up system resources.
cur.close()
cnx.close()

# Finally, output the total number of records processed and the file path.
count, str(out_jsonl)